# 3. Hierarchical Grid Index Benchmark: IJ vs. Z-Order (Morton)

Benchmarks comparing `HierarchicalGridIndexIJ` and `HierarchicalGridIndexZOrder`
across four query patterns on a `pred_patch` table with a B-tree index on `grid_cell_id`.

**Table schema**

| Column         | Type      | Description                                      |
|----------------|-----------|--------------------------------------------------|
| `id`           | BIGSERIAL | Primary key                                      |
| `embed_x`      | FLOAT     | X coordinate of the patch embedding              |
| `embed_y`      | FLOAT     | Y coordinate of the patch embedding              |
| `grid_cell_id` | BIGINT    | Hierarchical grid cell index (B-tree indexed)    |

**Benchmarks**

| # | Query |
|---|-------|
| 1 | First `pred_patch` for **1 grid cell** |
| 2 | First `pred_patch` for **1,000 grid cells** |
| 3 | **All** `pred_patches` for a single grid cell |
| 4 | **All** `pred_patches` for cells within a **bounding box** |

## Setup: Imports, Connection, Parameters

In [ ]:
import sys
import time
import random
import statistics

sys.path.insert(0, "/opt/PatchSorter/prototyping")
import psycopg2
from utils import HierarchicalGridIndexIJ, HierarchicalGridIndexZOrder

# ── Connection ────────────────────────────────────────────────────────────────
DATABASE_URL = "dbname=testdb user=testuser password=mypassword host=postgres-test"

# ── Tables ────────────────────────────────────────────────────────────────────
TABLE_IJ     = "proto_3_pred_patch_ij"
TABLE_ZORDER = "proto_3_pred_patch_zorder"

# ── Spatial parameters ────────────────────────────────────────────────────────
SPACE_SIZE = 1000.0   # coordinate space [0, SPACE_SIZE)
LEVEL      = 6        # resolution level; 2^6 = 64 cells per axis → 4 096 cells total
TOTAL_ROWS = 1_000_000
BATCH_SIZE = 100_000
COMMIT_FREQUENCY = 10

# ── Benchmark parameters ──────────────────────────────────────────────────────
RUNS = 10             # repetitions per benchmark for stable averages

# ── Index instances ───────────────────────────────────────────────────────────
indexes = {
    "IJ":      HierarchicalGridIndexIJ(cell_size=SPACE_SIZE),
    "Z-Order": HierarchicalGridIndexZOrder(cell_size=SPACE_SIZE),
}
tables = {
    "IJ":      TABLE_IJ,
    "Z-Order": TABLE_ZORDER,
}

conn = psycopg2.connect(DATABASE_URL)
cur  = conn.cursor()
cur.execute("SET work_mem = '512MB';")
conn.commit()
print("Connected.")
print(f"Level {LEVEL}: {2**LEVEL}x{2**LEVEL} = {(2**LEVEL)**2:,} cells, "
      f"~{TOTAL_ROWS // (2**LEVEL)**2} rows/cell on average")

## Phase 1: Create Tables & Insert Data

In [ ]:
def create_pred_patch_table(cur, conn, table_name: str) -> None:
    """Drop (if exists) and create a fresh pred_patch table with a
    B-tree index on grid_cell_id."""
    cur.execute(f"DROP TABLE IF EXISTS {table_name} CASCADE;")
    conn.commit()
    cur.execute(f"""
        CREATE TABLE {table_name} (
            id           BIGSERIAL    PRIMARY KEY,
            embed_x      FLOAT        NOT NULL,
            embed_y      FLOAT        NOT NULL,
            grid_cell_id BIGINT       NOT NULL
        );
    """)
    cur.execute(
        f"CREATE INDEX idx_{table_name}_cell "
        f"ON {table_name} (grid_cell_id);"
    )
    conn.commit()


for label in ("IJ", "Z-Order"):
    table = tables[label]
    index = indexes[label]
    scaled_size = SPACE_SIZE / (2 ** LEVEL)

    print(f"[{label}] Creating table '{table}' …")
    create_pred_patch_table(cur, conn, table)

    print(f"[{label}] Inserting {TOTAL_ROWS:,} rows …")
    rows_inserted = 0
    batch_count   = 0
    t0 = time.time()

    while rows_inserted < TOTAL_ROWS:
        this_batch = min(BATCH_SIZE, TOTAL_ROWS - rows_inserted)

        # Build rows in Python; psycopg2 executemany uses server-side
        # parameterised inserts which are fast for moderate batch sizes.
        rows = []
        for _ in range(this_batch):
            x = random.uniform(0, SPACE_SIZE)
            y = random.uniform(0, SPACE_SIZE)
            rows.append((x, y, index.point_to_cell(x, y, LEVEL)))

        cur.executemany(
            f"INSERT INTO {table} (embed_x, embed_y, grid_cell_id) "
            f"VALUES (%s, %s, %s)",
            rows,
        )
        rows_inserted += this_batch
        batch_count   += 1

        if batch_count % COMMIT_FREQUENCY == 0:
            conn.commit()
            elapsed = time.time() - t0
            rate    = rows_inserted / elapsed
            print(f"  {rows_inserted:>10,} / {TOTAL_ROWS:,}  "
                  f"({rate:,.0f} rows/s, ETA {(TOTAL_ROWS - rows_inserted) / rate:.0f}s)")

    conn.commit()
    cur.execute(f"ANALYZE {table};")
    conn.commit()

    elapsed = time.time() - t0
    print(f"[{label}] Done: {rows_inserted:,} rows in {elapsed:.2f}s "
          f"({rows_inserted / elapsed:,.0f} rows/s)\n")

## Phase 2: Benchmark Helpers

Shared utilities used across all four benchmarks.

In [ ]:
def run_benchmark(cur, query_fn, runs: int = RUNS) -> dict:
    """
    Execute query_fn(cur) `runs` times and return timing statistics (ms).
    query_fn must return the number of rows fetched so we can report it.
    """
    latencies = []
    row_counts = []
    for _ in range(runs):
        t0 = time.perf_counter()
        n_rows = query_fn(cur)
        latencies.append((time.perf_counter() - t0) * 1_000)
        row_counts.append(n_rows)

    return {
        "mean_ms":   statistics.mean(latencies),
        "median_ms": statistics.median(latencies),
        "min_ms":    min(latencies),
        "max_ms":    max(latencies),
        "stdev_ms":  statistics.stdev(latencies) if len(latencies) > 1 else 0.0,
        "rows":      statistics.mean(row_counts),
        "runs":      runs,
    }


def print_comparison(bm_name: str, results: dict) -> None:
    """Pretty-print a side-by-side comparison of IJ vs Z-Order results."""
    ij = results["IJ"]
    zo = results["Z-Order"]
    winner = "IJ" if ij["mean_ms"] <= zo["mean_ms"] else "Z-Order"
    speedup = max(ij["mean_ms"], zo["mean_ms"]) / min(ij["mean_ms"], zo["mean_ms"])

    print(f"\n{'─'*62}")
    print(f"  {bm_name}")
    print(f"{'─'*62}")
    print(f"  {'Metric':<18} {'IJ':>14} {'Z-Order':>14}")
    print(f"  {'─'*18} {'─'*14} {'─'*14}")
    for key, label in [
        ("mean_ms",   "Mean (ms)"),
        ("median_ms", "Median (ms)"),
        ("min_ms",    "Min (ms)"),
        ("max_ms",    "Max (ms)"),
        ("stdev_ms",  "Stdev (ms)"),
        ("rows",      "Avg rows"),
    ]:
        print(f"  {label:<18} {ij[key]:>14.3f} {zo[key]:>14.3f}")
    print(f"\n  ⟹  {winner} is faster by {speedup:.2f}×  (over {RUNS} runs)")
    print(f"{'─'*62}")


# Collect all results for the final summary table
all_results: dict[str, dict] = {}

scaled_size = SPACE_SIZE / (2 ** LEVEL)

# Pre-sample a pool of existing cell IDs from each table (used in benchmarks 1-3)
print("Sampling cell ID pools …")
cell_id_pools: dict[str, list[int]] = {}
for label, table in tables.items():
    cur.execute(f"""
        SELECT DISTINCT grid_cell_id
        FROM   {table}
        ORDER  BY random()
        LIMIT  2000;
    """)
    cell_id_pools[label] = [row[0] for row in cur.fetchall()]
    print(f"  [{label}] sampled {len(cell_id_pools[label])} distinct cell IDs")

print("Ready.")

## Benchmark 1 — First `pred_patch` for 1 grid cell

Each run picks a random cell ID from the pre-sampled pool and fetches
the first matching row using a `LIMIT 1` index scan.

In [ ]:
BM1_NAME = "BM1 — First pred_patch for 1 cell (LIMIT 1)"
bm1_results: dict[str, dict] = {}

for label, table in tables.items():
    pool = cell_id_pools[label]

    def query_bm1(cur, _table=table, _pool=pool):
        cell_id = random.choice(_pool)
        cur.execute(
            f"SELECT id, embed_x, embed_y FROM {_table} "
            f"WHERE grid_cell_id = %s LIMIT 1;",
            (cell_id,),
        )
        return len(cur.fetchall())   # always 0 or 1

    bm1_results[label] = run_benchmark(cur, query_bm1)
    print(f"[{label}] done")

print_comparison(BM1_NAME, bm1_results)
all_results[BM1_NAME] = bm1_results

## Benchmark 2 — First `pred_patch` for 1,000 grid cells

Each run selects 1,000 random cell IDs from the pool and issues a single
query with `LIMIT 1` per cell using `UNNEST` + a lateral join, so the
planner can use the B-tree index once per cell ID rather than doing a
sequential scan.

In [ ]:
BM2_NAME = "BM2 — First pred_patch for 1,000 cells (lateral LIMIT 1)"
bm2_results: dict[str, dict] = {}

for label, table in tables.items():
    pool = cell_id_pools[label]

    def query_bm2(cur, _table=table, _pool=pool):
        # Sample 1 000 distinct cell IDs for this run
        sample = random.sample(_pool, min(1_000, len(_pool)))
        cur.execute(
            f"""
            SELECT p.id, p.embed_x, p.embed_y
            FROM   UNNEST(%s::BIGINT[]) AS c(cell_id)
            JOIN LATERAL (
                SELECT id, embed_x, embed_y
                FROM   {_table}
                WHERE  grid_cell_id = c.cell_id
                LIMIT  1
            ) p ON TRUE;
            """,
            (sample,),
        )
        return len(cur.fetchall())

    bm2_results[label] = run_benchmark(cur, query_bm2)
    print(f"[{label}] done")

print_comparison(BM2_NAME, bm2_results)
all_results[BM2_NAME] = bm2_results

## Benchmark 3 — All `pred_patches` for a single grid cell

Each run picks a random cell ID and fetches every row that belongs to it.
At ~244 rows/cell on average this exercises a multi-row index range scan.

In [ ]:
BM3_NAME = "BM3 — All pred_patches for 1 cell (full index range scan)"
bm3_results: dict[str, dict] = {}

for label, table in tables.items():
    pool = cell_id_pools[label]

    def query_bm3(cur, _table=table, _pool=pool):
        cell_id = random.choice(_pool)
        cur.execute(
            f"SELECT id, embed_x, embed_y FROM {_table} "
            f"WHERE grid_cell_id = %s;",
            (cell_id,),
        )
        return len(cur.fetchall())

    bm3_results[label] = run_benchmark(cur, query_bm3)
    print(f"[{label}] done  (avg {bm3_results[label]['rows']:.1f} rows/run)")

print_comparison(BM3_NAME, bm3_results)
all_results[BM3_NAME] = bm3_results

## Benchmark 4 — All `pred_patches` for cells within a bounding box

Each run picks a random axis-aligned bounding box covering ~10 % of the
space (100 × 100 units out of 1 000 × 1 000, or roughly 6 × 6 = 36 cells
at level 6) and fetches every matching row.

Both index types receive the **same list of cell IDs** computed from the
bbox, so the query structure is identical.  The Z-order advantage is
structural: Morton codes for spatially adjacent cells are numerically
close, so the B-tree index scan visits fewer leaf pages and produces
better I/O locality than the scattered IJ codes.

In [ ]:
BM4_NAME = "BM4 — All pred_patches for cells in a bounding box (~36 cells)"
bm4_results: dict[str, dict] = {}

# Box side length: ~10 % of the space gives ~6×6 = 36 cells at level 6
BOX_SIDE = SPACE_SIZE * 0.1   # 100 units out of 1 000
cell_w   = SPACE_SIZE / (2 ** LEVEL)  # cell width at LEVEL (~15.6 units)


def bbox_cell_ids(index, x_min: float, y_min: float,
                  x_max: float, y_max: float, level: int) -> list[int]:
    """
    Return every cell ID that overlaps the axis-aligned bounding box
    [x_min, x_max) × [y_min, y_max) at the given level.
    Works for both IJ and Z-order encodings.
    """
    cw = SPACE_SIZE / (2 ** level)
    i_lo = int(x_min // cw)
    i_hi = int((x_max - 1e-10) // cw)
    j_lo = int(y_min // cw)
    j_hi = int((y_max - 1e-10) // cw)

    # Sample a representative point in each (i, j) tile → same API for both
    return [
        index.point_to_cell((i + 0.5) * cw, (j + 0.5) * cw, level)
        for i in range(i_lo, i_hi + 1)
        for j in range(j_lo, j_hi + 1)
    ]


for label, table in tables.items():
    index = indexes[label]

    def query_bm4(cur, _table=table, _index=index):
        # Random bbox origin so the box stays fully inside [0, SPACE_SIZE)
        x_min = random.uniform(0, SPACE_SIZE - BOX_SIDE)
        y_min = random.uniform(0, SPACE_SIZE - BOX_SIDE)
        x_max = x_min + BOX_SIDE
        y_max = y_min + BOX_SIDE

        cell_ids = bbox_cell_ids(_index, x_min, y_min, x_max, y_max, LEVEL)

        cur.execute(
            f"SELECT id, embed_x, embed_y "
            f"FROM   {_table} "
            f"WHERE  grid_cell_id = ANY(%s::BIGINT[]);",
            (cell_ids,),
        )
        return len(cur.fetchall())

    bm4_results[label] = run_benchmark(cur, query_bm4)
    n_cells = len(bbox_cell_ids(index, 0, 0, BOX_SIDE, BOX_SIDE, LEVEL))
    print(f"[{label}] done  "
          f"(~{n_cells} cells/bbox, avg {bm4_results[label]['rows']:.0f} rows/run)")

print_comparison(BM4_NAME, bm4_results)
all_results[BM4_NAME] = bm4_results

## Summary

In [ ]:
COL_W = 12

header = f"  {'Benchmark':<45} {'IJ mean':>{COL_W}} {'ZO mean':>{COL_W}} {'Winner':>{COL_W}} {'Speedup':>{COL_W}}"
print(f"\n{'═'*len(header)}")
print("  BENCHMARK SUMMARY — IJ vs Z-Order")
print(f"{'═'*len(header)}")
print(header)
print(f"  {'─'*45} {'─'*COL_W} {'─'*COL_W} {'─'*COL_W} {'─'*COL_W}")

for bm_name, results in all_results.items():
    ij_ms = results["IJ"]["mean_ms"]
    zo_ms = results["Z-Order"]["mean_ms"]
    winner  = "IJ" if ij_ms <= zo_ms else "Z-Order"
    speedup = max(ij_ms, zo_ms) / min(ij_ms, zo_ms)
    short   = bm_name[:45]
    print(f"  {short:<45} {ij_ms:>{COL_W}.3f} {zo_ms:>{COL_W}.3f} {winner:>{COL_W}} {speedup:>{COL_W}.2f}×")

print(f"{'═'*len(header)}")
print(f"  All timings in milliseconds (mean over {RUNS} runs).")

cur.close()
conn.close()
print("Connection closed.")